# Build PF-CLM TWS by HUC2 (WY2003)

Reads CONUS2.1 ParFlow/CLM daily-average PFBs from `/hydrodata`, computes
Total Water Storage (TWS), and aggregates to 18 CONUS HUC2 basins as monthly means.

**Environment**: Verde / OpenOnDemand, direct `/hydrodata` access  
**Runtime**: ~25 min (364 days of PFB I/O)  
**Output**: `pfclm_huc2_tws.nc` — monthly TWS anomalies in mm and cm EWH

### TWS formula (per grid cell, per day)
```
TWS_mm = (SUBstorage + SURF_WATstorage) / CELL_AREA * 1000 + SWE
```
- `SUBstorage`: column-summed subsurface storage (m³ per cell)
- `SURF_WATstorage`: ponded surface water (m³ per cell)
- `SWE`: CLM snow water equivalent (mm)
- `CELL_AREA`: 1e6 m² (1km × 1km LCC grid)

### Sections
1. Environment check
2. Configuration & imports
3. Load HUC2 mask
4. Spot-check PFB files
5. Build daily TWS & aggregate to HUC2
6. Resample to monthly means
7. Compute anomalies & save
8. Sanity-check plot

## 1. Environment check

In [ ]:
import importlib, sys

required = {
    'numpy':          'numpy',
    'xarray':         'xarray',
    'pandas':         'pandas',
    'matplotlib':     'matplotlib',
    'hf_hydrodata':   'hf_hydrodata',
    'parflow.tools.io': 'pftools (parflow)',
}

ok = True
for mod, label in required.items():
    try:
        importlib.import_module(mod)
        print(f'  OK  {label}')
    except ImportError:
        print(f'  MISSING  {label}  ->  pip install {mod.split(".")[0]}')
        ok = False

print()
print('Python:', sys.version)
if ok:
    print('All dependencies present.')
else:
    print('Install missing packages before continuing.')

## 2. Configuration & imports

**Edit paths here if needed** — everything else reads from these variables.

In [ ]:
from pathlib import Path
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import hf_hydrodata as hf
from parflow.tools.io import read_pfb

# ── Paths ─────────────────────────────────────────────────────────────────────
DAILY_DIR = Path('/hydrodata/temp/CONUS2.1/WY2003_V4/averages/daily')
OUT_DIR   = Path('output')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Domain constants (CONUS2.1) ─────────────────────────────────────────────────
NX, NY, NZ = 4442, 3256, 10
DX = DY    = 1000.0       # metres (LCC projection, effectively equal-area)
CELL_AREA  = DX * DY      # m2
WATER_YEAR = 2003
N_DAYS     = 364           # Sep 30 (day 365) is missing from the daily PFBs
WY_DATES   = pd.date_range('2002-10-01', periods=N_DAYS, freq='D')

# ── HUC2 labels ───────────────────────────────────────────────────────────
HUC2_NAMES = {
    1:  'New England',    2:  'Mid-Atlantic',    3:  'South Atlantic',
    4:  'Great Lakes',    5:  'Ohio',             6:  'Tennessee',
    7:  'Upper MS',       8:  'Lower MS',         9:  'Souris-Red-Rainy',
    10: 'Missouri',       11: 'Arkansas-White',   12: 'Texas Gulf',
    13: 'Rio Grande',     14: 'Upper Colorado',   15: 'Lower Colorado',
    16: 'Great Basin',    17: 'Pacific NW',       18: 'California',
}
HUC2_IDS = sorted(HUC2_NAMES.keys())

print('Config loaded.')
print(f'  DAILY_DIR exists : {DAILY_DIR.exists()}')

## 3. Load HUC2 mask on CONUS2 grid

Uses `hf_hydrodata` to get the HUC2 integer raster on the CONUS2 1km LCC grid.
Cells are effectively equal-area on LCC, so aggregation is a simple arithmetic mean.

In [ ]:
huc2_mask = hf.get_gridded_data({
    'dataset': 'huc_mapping',
    'grid':    'conus2',
    'level':   '2',
}).astype(int).squeeze()   # (NY, NX)

print(f'CONUS2 HUC2 mask shape : {huc2_mask.shape}  (expect {NY} x {NX})')
print(f'HUC IDs present        : {sorted(np.unique(huc2_mask[huc2_mask > 0]))}')
print('Cells per HUC2:')
for hid in HUC2_IDS:
    print(f'  HUC{hid:02d} {HUC2_NAMES[hid]:20s}  {int((huc2_mask == hid).sum()):7d} cells')

## 4. Spot-check daily PFB files

In [ ]:
test_days = [1, 90, 180, 270, 365]
print('Spot-checking daily PFB files ...')
for d in test_days:
    tag = str(d).rjust(3, '0')
    for var in ['SUBstorage', 'SURF_WATstorage', 'swe_out']:
        fpath  = DAILY_DIR / f'{var}.{WATER_YEAR}.daily.{tag}.pfb'
        status = 'OK' if fpath.exists() else 'MISSING'
        print(f'  [{status}] day {tag}  {var}')

print('\nTest read of day 001 ...')
sub  = read_pfb(str(DAILY_DIR / f'SUBstorage.{WATER_YEAR}.daily.001.pfb'))
surf = read_pfb(str(DAILY_DIR / f'SURF_WATstorage.{WATER_YEAR}.daily.001.pfb'))
swe  = read_pfb(str(DAILY_DIR / f'swe_out.{WATER_YEAR}.daily.001.pfb'))
print(f'  SUBstorage shape       : {sub.shape}  (expect {NZ},{NY},{NX})')
print(f'  SURF_WATstorage shape  : {surf.shape}')
print(f'  swe_out shape          : {swe.shape}')
print(f'  SUBstorage total range : {sub.sum(axis=0).min():.1f} - {sub.sum(axis=0).max():.1f} m3')
print(f'  SWE range              : {np.squeeze(swe).min():.1f} - {np.squeeze(swe).max():.1f} mm')

## 5. Build daily TWS and aggregate to HUC2

Reads each day's PFBs, computes per-cell TWS, and accumulates HUC2 means on the fly.
This avoids building the full (364, 3256, 4442) stack in memory (~21 GB).

**Note**: SWE nodata (−9999) is set to 0 before averaging. This may slightly bias
coastal/edge basins where HUC2 mask cells fall outside the ParFlow active domain.

In [ ]:
import time
t0 = time.time()

mask_flat = huc2_mask.ravel()
daily_huc2 = np.zeros((N_DAYS, len(HUC2_IDS)), dtype=np.float32)

for i in range(N_DAYS):
    tag  = str(i + 1).rjust(3, '0')
    sub  = read_pfb(str(DAILY_DIR / f'SUBstorage.{WATER_YEAR}.daily.{tag}.pfb'))
    surf = np.squeeze(read_pfb(str(DAILY_DIR / f'SURF_WATstorage.{WATER_YEAR}.daily.{tag}.pfb')))
    swe  = np.squeeze(read_pfb(str(DAILY_DIR / f'swe_out.{WATER_YEAR}.daily.{tag}.pfb')))
    swe  = np.where(swe < -9000, 0.0, swe)
    tws  = (sub.sum(axis=0) + surf) / CELL_AREA * 1000.0 + swe  # (NY, NX) mm EWH
    tws_flat = tws.ravel()
    for j, hid in enumerate(HUC2_IDS):
        idx = np.where(mask_flat == hid)[0]
        daily_huc2[i, j] = tws_flat[idx].mean()
    if i % 30 == 0:
        print(f'  Day {i+1}/{N_DAYS}  ({WY_DATES[i].date()})')

print(f'Done in {time.time()-t0:.1f}s')
print(f'daily_huc2 shape: {daily_huc2.shape},  memory: {daily_huc2.nbytes/1e6:.1f} MB')

## 6. Resample to monthly means

In [ ]:
tws_huc2_daily = xr.DataArray(
    daily_huc2.T,   # (huc2_id, time)
    dims=('huc2_id', 'time'),
    coords={'huc2_id': HUC2_IDS, 'time': WY_DATES},
    attrs={'units': 'mm EWH'},
)
tws_monthly = tws_huc2_daily.resample(time='MS').mean()
print(f'Monthly timesteps: {[str(t)[:7] for t in tws_monthly.time.values]}')

## 7. Compute anomalies & save

Anomalies are computed by subtracting the WY2003 annual mean per HUC2.
This is a different reference than GRACE products (which use a ~2004–2009 baseline),
so only temporal patterns (correlations) are directly comparable, not absolute offsets.

In [ ]:
pf_ds = xr.Dataset(
    {'tws_mm': tws_monthly},
    coords={'huc2_name': ('huc2_id', [HUC2_NAMES[h] for h in HUC2_IDS])}
)

# Rebase to WY2003 annual mean
pf_ds['tws_mm'] = pf_ds['tws_mm'] - pf_ds['tws_mm'].mean('time')
pf_ds['tws_cm'] = pf_ds['tws_mm'] / 10.0
pf_ds['tws_cm'].attrs = {'units': 'cm EWH', 'long_name': 'TWS anomaly re WY2003 mean'}

pf_ds.to_netcdf(OUT_DIR / 'pfclm_huc2_tws.nc')
print(f'Saved -> {OUT_DIR}/pfclm_huc2_tws.nc')
print(pf_ds)

print('\nPF-CLM TWS range by HUC2 (cm EWH):')
for hid in HUC2_IDS:
    ts = pf_ds['tws_cm'].sel(huc2_id=hid).values
    print(f'  HUC{hid:02d} {HUC2_NAMES[hid]:20s}  min={ts.min():6.1f}  max={ts.max():6.1f}  range={ts.max()-ts.min():5.1f}')

## 8. Sanity-check plot

Quick 3×6 panel of monthly TWS anomalies by HUC2.

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(18, 9), constrained_layout=True)
fig.suptitle('PF-CLM WY2003 TWS Anomaly by HUC2', fontweight='bold', fontsize=12)

for i, hid in enumerate(HUC2_IDS):
    ax = axes.ravel()[i]
    ts = pf_ds['tws_cm'].sel(huc2_id=hid)
    ax.plot(ts.time.values, ts.values, color='#1A7C1A', lw=2)
    ax.axhline(0, color='k', lw=0.5, ls='--', alpha=0.4)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[10, 1, 4, 7]))
    ax.set_title(f'HUC{hid:02d}: {HUC2_NAMES[hid]}', fontsize=8, fontweight='bold')
    ax.set_ylabel('\u0394TWS (cm EWH)', fontsize=7)
    ax.tick_params(labelsize=7)

plt.savefig(OUT_DIR / 'pfclm_wy2003_all_huc2.png', dpi=150, bbox_inches='tight')
print(f'Saved -> {OUT_DIR}/pfclm_wy2003_all_huc2.png')
plt.show()